# 04 — Spatial Binning (Small-Scale Testing)

Computes spatial co-occurrence between the top 50 plant species and all
pollinator species at 0.5° resolution. Identifies the top 20 overlap bins
by combined observation density.

**Overlap score used here:** `(plant_count / max_plant_count) × (pollinator_count / max_pollinator_count)`

This is a simple density product — it selects bins where both sides are
well-observed. As noted in the Jaccard notebook (06), this score is heavily
biased toward high-observation-density areas (urban regions, accessible parks)
rather than ecologically meaningful co-occurrence. The Jaccard index in
notebook 06 was developed as a corrective measure.

**Output:** `top20_overlap_bins.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from pathlib import Path

BASE       = Path("/scratch/ariana.l")
PLANTS_IN  = BASE / "Plant Pollinator Initial Analysis" / "plant_flowering_events.parquet"
POLL_IN    = BASE / "Plant Pollinator Initial Analysis" / "pollinator_observations_v2.csv"
OUT_DIR    = BASE

BIN_SIZE   = 0.5

print("Paths OK")

In [ ]:
# Load data
plants = pd.read_parquet(PLANTS_IN)
pollinators = pd.read_csv(POLL_IN, low_memory=False)

print(f"Plants: {len(plants):,} records, {plants['species'].nunique()} species")
print(f"Pollinators: {len(pollinators):,} records, {pollinators['pollinator_species'].nunique():,} species")

In [ ]:
# Add spatial bins
plants['lat_bin'] = (np.floor(plants['lat'] / BIN_SIZE) * BIN_SIZE).round(1)
plants['lon_bin'] = (np.floor(plants['lon'] / BIN_SIZE) * BIN_SIZE).round(1)
pollinators['lat_bin'] = (np.floor(pollinators['lat'] / BIN_SIZE) * BIN_SIZE).round(1)
pollinators['lon_bin'] = (np.floor(pollinators['lon'] / BIN_SIZE) * BIN_SIZE).round(1)

plant_bins = plants.groupby(['lat_bin', 'lon_bin']).size().reset_index(name='count')
pol_bins = pollinators.groupby(['lat_bin', 'lon_bin']).size().reset_index(name='count')

print(f"Plant bins: {len(plant_bins):,}")
print(f"Pollinator bins: {len(pol_bins):,}")

In [ ]:
# Compute overlap score and identify top 20 bins
# Score = normalized plant density × normalized pollinator density
# NOTE: this score is biased toward high-recorder-effort areas.
# It selects where both observers and pollinators were active,
# not necessarily where ecologically meaningful interactions occur.
overlap = plant_bins.merge(pol_bins, on=['lat_bin', 'lon_bin'], suffixes=('_plant', '_pol'))
overlap['score'] = (
    (overlap['count_plant'] / overlap['count_plant'].max()) *
    (overlap['count_pol'] / overlap['count_pol'].max())
)
top20 = overlap.sort_values('score', ascending=False).head(20)

print(f"Shared bins: {len(overlap):,}")
print(f"\nTop 20 overlap bins:")
print(top20[['lat_bin', 'lon_bin', 'count_plant', 'count_pol', 'score']].to_string(index=False))

In [ ]:
# Save
out_path = OUT_DIR / "top20_overlap_bins.csv"
top20.rename(columns={'lat_bin': 'lat_min', 'lon_bin': 'lon_min'}).to_csv(out_path, index=False)
print(f"Saved → {out_path}")

In [ ]:
# Visualization
plant_norm = Normalize(vmin=0, vmax=plant_bins['count'].max())
pol_norm   = Normalize(vmin=0, vmax=pol_bins['count'].max())

fig, ax = plt.subplots(figsize=(16, 9))

for _, row in plant_bins.iterrows():
    ax.add_patch(plt.Rectangle(
        (row['lon_bin'], row['lat_bin']), BIN_SIZE, BIN_SIZE,
        color=plt.cm.Greens(plant_norm(row['count'])), alpha=0.5
    ))
for _, row in pol_bins.iterrows():
    ax.add_patch(plt.Rectangle(
        (row['lon_bin'], row['lat_bin']), BIN_SIZE, BIN_SIZE,
        color=plt.cm.Reds(pol_norm(row['count'])), alpha=0.5
    ))
for _, row in top20.iterrows():
    ax.add_patch(plt.Circle(
        (row['lon_bin'] + BIN_SIZE/2, row['lat_bin'] + BIN_SIZE/2),
        BIN_SIZE/2 * 1.1, fill=False, edgecolor='blue', linewidth=2
    ))

ax.set_xlim(-130, -60)
ax.set_ylim(24, 50)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Plant (green) vs Pollinator (red) Density — Top 20 overlap bins circled\n'
             'Note: density is a proxy for observer effort, not true abundance')

plt.colorbar(plt.cm.ScalarMappable(cmap='Greens', norm=plant_norm), ax=ax,
             location='left', shrink=0.6, pad=0.08, label='Plant count')
plt.colorbar(plt.cm.ScalarMappable(cmap='Reds', norm=pol_norm), ax=ax,
             location='right', shrink=0.6, pad=0.01, label='Pollinator count')

plt.tight_layout()
plt.savefig(OUT_DIR / 'overlap_map_top20.png', dpi=150)
print("Saved.")